## "Анализ эффективности маркетинговых кампаний в социальных сетях для выявления закономерностей вовлечения аудитории".

### 2.1. Проанализировать цели проекта. С точки зрения бизнеса, определить бизнес-цели и критерии их достижения, с точки зрения науки, задать границы исследования в рамках выбранной темы, описать объект и предмет исследования, сформулировать аналитическую гипотезу.

**Бизнес-проблема.**

Высокие показатели охвата, кликов и вовлечённости не всегда означают эффективное использование рекламного бюджета. Поэтому необходимо определить, какие характеристики рекламных кампаний и аудитории связаны с более высокой финансовой отдачей.

**Бизнес-цель проекта.**

Повысить эффективность расходования рекламного бюджета за счёт выявления наиболее результативных маркетинговых стратегий, категорий товаров и сегментов покупателей.

**Критерии достижения бизнес-цели.**

Предварительно цель считается достигнутой, если применение результатов анализа позволит:

* увеличить финансовую отдачу от рекламных расходов не менее чем на 10%
* снизить стоимость привлечения или целевого действия не менее чем на 10%
* выявить категории товаров, покупателей и маркетинговые стратегии, финансовая эффективность которых превышает средний уровень не менее чем на 15%
* определить направления для перераспределения не менее 15% рекламного бюджета из менее эффективных кампаний в более эффективные

**Практическая ценность.**

Результаты исследования могут использоваться для выявления неэффективных рекламных расходов, выбора наиболее результативных маркетинговых стратегий и поддержки решений по перераспределению рекламного бюджета между категориями товаров и сегментами покупателей.

**Объект исследования.**

Онлайн-маркетинговые кампании по продвижению товаров среди различных сегментов покупателей.

**Предмет исследования.**

Алгоритмы интеллектуального анализа данных и машинного обучения для оценки финансовой эффективности онлайн-маркетинговых кампаний и выявления наиболее результативных маркетинговых стратегий.

**Границы исследования.**

Исследование ограничивается онлайн-маркетинговыми кампаниями по продвижению товаров, для которых доступны сведения о рекламных расходах, результатах кампании, категориях товаров и характеристиках покупателей. Предполагается анализировать кампании в социальных сетях и других цифровых рекламных каналах. Конкретные платформы, категории товаров, сегменты покупателей и временной период будут уточнены после окончательного выбора источников данных.

**Исследовательский вопрос.**

Какие характеристики рекламных кампаний, каналов и аудитории связаны с различиями в финансовой эффективности рекламы?

**Гипотеза.**

H1: ROAS статистически значимо различается хотя бы между платформами, типами маркетинговых кампаний или сегментами аудитории. 

H0: ROAS статистически не связан характеристиками рекламных и маркетинговых кампаний, платформами или сегментами аудитории, а их использование не позволяет улучшить прогноз финансовой эффективности.

**2.2. Подбор и изучение источников данных**

Для исследования выбраны четыре тематических фрагмента данных. Они описывают финансовую эффективность рекламных кампаний, количественную вовлечённость аудитории видеоконтента, текстовую реакцию пользователей и внешний тематический контекст цифрового маркетинга. Источники анализируются раздельно. Объединение выполняется только там, где присутствует содержательный идентификатор или вложенная структура.

| Фрагмент | Характеризуемая область | Формат и способ получения |
|---|---|---|
| Платная рекламная кампания | расходы, доход, конверсии и эффективность рекламы | CSV, программная загрузка публичного файла |
| YouTube Trending Video Statistics | просмотры, реакции и комментарии аудитории | CSV и JSON, программная загрузка публичных файлов |
| Twitter US Airline Sentiment | текстовая реакция и тональность сообщений | CSV, программная загрузка публичного файла |
| Тематический контекст digital marketing | поведение посетителей интернет-магазина и публикации о digital marketing | CSV через UCI Repository и JSON через OpenAlex API |

In [1]:
from pathlib import Path
import io
import json
import zipfile
import time
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from IPython.display import display

DATA_DIR = Path("course_data")
DATA_DIR.mkdir(exist_ok=True)


def download_kaggle_file(dataset_ref, filename, local_name=None):
    """Скачивает один публичный файл с Kaggle и использует локальный кеш."""
    local_path = DATA_DIR / (local_name or Path(filename).name)
    if local_path.exists() and local_path.stat().st_size > 0:
        return local_path
    url = (
        "https://www.kaggle.com/api/v1/datasets/download/"
        f"{dataset_ref}?filename={requests.utils.quote(filename, safe='')}"
    )
    response = requests.get(url, timeout=180)
    response.raise_for_status()
    content = response.content
    if content[:2] == b"PK":
        with zipfile.ZipFile(io.BytesIO(content)) as archive:
            names = archive.namelist()
            if not names:
                raise ValueError(f"Пустой архив: {filename}")
            requested = Path(filename).name
            selected = next((n for n in names if Path(n).name == requested), names[0])
            content = archive.read(selected)
    local_path.write_bytes(content)
    return local_path


def download_url(url, local_name):
    """Скачивает публичный API-ответ и кеширует его локально."""
    local_path = DATA_DIR / local_name
    if local_path.exists() and local_path.stat().st_size > 0:
        return local_path
    response = requests.get(
        url,
        headers={"User-Agent": "coursework-data-research/1.0"},
        timeout=180,
    )
    response.raise_for_status()
    local_path.write_bytes(response.content)
    return local_path


def show_table_profile(df, name, head_rows=3):
    print(f"{name}: shape={df.shape}")
    print("Столбцы:", list(df.columns))
    print("Типы данных:")
    display(df.dtypes.rename("dtype").to_frame())
    print("Пропуски:")
    display(df.isna().sum().rename("missing").to_frame())
    print(f"Первые {head_rows} строки:")
    display(df.head(head_rows))

**Фрагмент 1. Финансовая эффективность платной рекламы**

Shopping Mall Paid Search Campaign Dataset используется как основной источник для проверки финансовой части гипотезы. Набор содержит расходы, доход, конверсии и показатели рекламных групп. По полям `Revenue` и `Cost` может быть рассчитан ROAS. При интерпретации отдельно учитываются `Revenue`, `Sale Amount` и `P&L`, поскольку их экономический смысл необходимо проверить по фактическим значениям и описанию источника.

In [2]:
campaign_path = download_kaggle_file(
    "marceaxl82/shopping-mall-paid-search-campaign-dataset",
    "final_shop_6modata.csv",
    "campaign.csv",
)
campaign = pd.read_csv(campaign_path)
show_table_profile(campaign, "Платная рекламная кампания")
print("Уникальные рекламные группы:", campaign["Ad Group"].nunique())
print("Месяцы:", sorted(campaign["Month"].dropna().unique().tolist()))
print("Диапазон Cost:", (campaign["Cost"].min(), campaign["Cost"].max()))
print("Диапазон Revenue:", (campaign["Revenue"].min(), campaign["Revenue"].max()))

campaign_profile = campaign.assign(
    roas=lambda x: x["Revenue"] / x["Cost"].where(x["Cost"] != 0),
    ctr_recalculated=lambda x: x["Clicks"] / x["Impressions"].where(x["Impressions"] != 0),
    cost_band=lambda x: pd.qcut(x["Cost"], q=3, labels=["low", "medium", "high"], duplicates="drop"),
)
print("Минимальный профиль финансовых показателей:")
display(campaign_profile[["Cost", "Revenue", "P&L", "roas", "CTR", "CPC", "Conv Rate"]].describe().T)

Платная рекламная кампания: shape=(190, 12)
Столбцы: ['Ad Group', 'Month', 'Impressions', 'Clicks', 'CTR', 'Conversions', 'Conv Rate', 'Cost', 'CPC', 'Revenue', 'Sale Amount', 'P&L']
Типы данных:


,dtype
Ad Group,str
Month,str
Impressions,int64
Clicks,int64
CTR,float64
Conversions,int64
Conv Rate,float64
Cost,int64
CPC,float64
Revenue,int64


Пропуски:


,missing
Ad Group,0
Month,0
Impressions,0
Clicks,0
CTR,0
Conversions,0
Conv Rate,0
Cost,0
CPC,0
Revenue,0


Первые 3 строки:


,Ad Group,Month,Impressions,Clicks,CTR,Conversions,Conv Rate,Cost,CPC,Revenue,Sale Amount,P&L
0,Shop - 1:1 - Desk - [shop coupon code],July,16038,6504,0.41,1166,0.10,6669,1.03,6402,136770.05,-267.086
1,Shop - 1:1 - Desk - [shop coupon],July,36462,14367,0.39,2188,0.09,13746,0.96,13262,283215.21,-483.951
2,Shop - 1:1 - Desk - [shop discount code],July,3635,1458,0.40,248,0.09,1606,1.10,1723,39165.46,117.136


Уникальные рекламные группы: 40
Месяцы: ['August', 'July', 'November', 'October', 'September']
Диапазон Cost: (np.int64(1), np.int64(43542))
Диапазон Revenue: (np.int64(0), np.int64(42440))
Минимальный профиль финансовых показателей:


,count,mean,std,min,25%,50%,75%,max
Cost,190.0,3344.063158,6524.606753,1.000,188.500000,563.500000,2967.000000,43542.000
Revenue,190.0,2957.684211,5962.413097,0.000,144.500000,459.500000,2672.250000,42440.000
P&L,190.0,-386.361916,903.073776,-5672.271,-294.977500,-75.978000,-6.027500,1507.685
roas,190.0,1.032206,1.830349,0.000,0.659233,0.848149,0.967513,20.000
CTR,190.0,0.272105,0.107894,0.050,0.192500,0.285000,0.360000,0.470
CPC,190.0,0.791263,0.403312,0.140,0.480000,0.635000,1.107500,2.020
Conv Rate,190.0,0.079737,0.052859,0.000,0.050000,0.070000,0.100000,0.500


Существенные признаки первого фрагмента относятся к следующим шкалам: `Ad Group` является категориальным признаком, `Month` характеризует календарный период и относится к интервальной шкале, `Cost`, `Revenue`, `P&L`, `CTR`, `CPC` и `Conv Rate` относятся к шкале отношений, а `Impressions`, `Clicks` и `Conversions` являются счётными величинами абсолютной шкалы. Производный `cost_band` является порядковым признаком и используется только для описательного сравнения, а не как исходное поле кампании.

**Фрагмент 2. Количественная вовлечённость аудитории видеоконтента**

Trending YouTube Video Statistics используется для анализа реакций аудитории на видеоконтент. CSV содержит наблюдения по видео, а JSON содержит иерархический справочник категорий. Связь между двумя файлами выполняется по `category_id`. Этот фрагмент не содержит рекламных расходов, поэтому используется как дополнительный источник факторов вовлечённости.

In [3]:
youtube_csv_path = download_kaggle_file(
    "datasnaek/youtube-new", "USvideos.csv", "USvideos.csv"
)
youtube_json_path = download_kaggle_file(
    "datasnaek/youtube-new", "US_category_id.json", "US_category_id.json"
)
youtube = pd.read_csv(youtube_csv_path)
show_table_profile(youtube, "YouTube US videos")

category_data = json.loads(youtube_json_path.read_text(encoding="utf-8"))
print("Тип корневого JSON-объекта:", type(category_data).__name__)
print("Ключи верхнего уровня:", list(category_data.keys()))
print("Количество объектов items:", len(category_data["items"]))
category_table = pd.json_normalize(category_data["items"])
print("Структура справочника категорий:")
display(category_table.head(3))
print("Распределение просмотров, лайков и комментариев:")
display(youtube[["views", "likes", "comment_count"]].describe().T)

YouTube US videos: shape=(40949, 16)
Столбцы: ['video_id', 'trending_date', 'title', 'channel_title', 'category_id', 'publish_time', 'tags', 'views', 'likes', 'dislikes', 'comment_count', 'thumbnail_link', 'comments_disabled', 'ratings_disabled', 'video_error_or_removed', 'description']
Типы данных:


,dtype
video_id,str
trending_date,str
title,str
channel_title,str
category_id,int64
publish_time,str
tags,str
views,int64
likes,int64
dislikes,int64


Пропуски:


,missing
video_id,0
trending_date,0
title,0
channel_title,0
category_id,0
publish_time,0
tags,0
views,0
likes,0
dislikes,0


Первые 3 строки:


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description
0,2kyS6SvSYSE,17.14.11,WE WANT TO TALK ABOUT OUR MARRIAGE,CaseyNeistat,22,2017-11-13T17:13:01.000Z,SHANtell martin,748374,57527,2966,15954,https://i.ytimg.com/vi/2kyS6SvSYSE/default.jpg,False,False,False,SHANTELL'S CHANNEL - https://www.youtube.com/s...
1,1ZAPwfrtAFY,17.14.11,The Trump Presidency: Last Week Tonight with J...,LastWeekTonight,24,2017-11-13T07:30:00.000Z,"last week tonight trump presidency|""last week ...",2418783,97185,6146,12703,https://i.ytimg.com/vi/1ZAPwfrtAFY/default.jpg,False,False,False,"One year after the presidential election, John..."
2,5qpjK5DgCt4,17.14.11,"Racist Superman | Rudy Mancuso, King Bach & Le...",Rudy Mancuso,23,2017-11-12T19:05:24.000Z,"racist superman|""rudy""|""mancuso""|""king""|""bach""...",3191434,146033,5339,8181,https://i.ytimg.com/vi/5qpjK5DgCt4/default.jpg,False,False,False,WATCH MY PREVIOUS VIDEO ▶ \n\nSUBSCRIBE ► http...


Тип корневого JSON-объекта: dict
Ключи верхнего уровня: ['kind', 'etag', 'items']
Количество объектов items: 32
Структура справочника категорий:


,kind,etag,id,snippet.channelId,snippet.title,snippet.assignable
0,youtube#videoCategory,"""m2yskBQFythfE4irbTIeOgYYfBU/Xy1mB4_yLrHy_BmKm...",1,UCBR8-60-B28hp2BmDPdntcQ,Film & Animation,True
1,youtube#videoCategory,"""m2yskBQFythfE4irbTIeOgYYfBU/UZ1oLIIz2dxIhO45Z...",2,UCBR8-60-B28hp2BmDPdntcQ,Autos & Vehicles,True
2,youtube#videoCategory,"""m2yskBQFythfE4irbTIeOgYYfBU/nqRIq97-xe5XRZTxb...",10,UCBR8-60-B28hp2BmDPdntcQ,Music,True


Распределение просмотров, лайков и комментариев:


,count,mean,std,min,25%,50%,75%,max
views,40949.0,2.360785e+06,7.394114e+06,549.0,242329.0,681861.0,1823157.0,225211923.0
likes,40949.0,7.426670e+04,2.288853e+05,0.0,5424.0,18091.0,55417.0,5613827.0
comment_count,40949.0,8.446804e+03,3.743049e+04,0.0,614.0,1856.0,5755.0,1361580.0


Для второго фрагмента `video_id` является номинальной шкалой ID, `title`, `tags`, `description` и URL миниатюры относятся к контентной шкале, `channel_title` и категория видео являются категориальными признаками, `trending_date` и `publish_time` относятся к интервальной шкале, а `views`, `likes`, `dislikes` и `comment_count` являются абсолютными величинами. Поля `comments_disabled`, `ratings_disabled` и `video_error_or_removed` имеют дихотомическую шкалу.

**Фрагмент 3. Текстовая реакция пользователей**

Twitter US Airline Sentiment используется как открытый текстовый фрагмент для изучения пользовательской реакции. Он не описывает рекламные расходы напрямую, но позволяет показать обработку неструктурированных текстовых данных, аннотирование тональности и подготовку признаков для последующей модели TF-IDF и Logistic Regression.

In [4]:
airline_path = download_kaggle_file(
    "crowdflower/twitter-airline-sentiment", "Tweets.csv", "Tweets.csv"
)
airline = pd.read_csv(airline_path)
show_table_profile(airline, "Twitter US Airline Sentiment")
print("Уникальные авиакомпании:", airline["airline"].nunique())
print("Распределение тональности:")
display(airline["airline_sentiment"].value_counts().rename("count").to_frame())

airline_profile = airline.assign(
    text_length=airline["text"].fillna("").str.len(),
    word_count=airline["text"].fillna("").str.split().str.len(),
)
print("Минимальный числовой профиль текста и метаданных:")
display(airline_profile[["airline_sentiment_confidence", "negativereason_confidence", "retweet_count", "text_length", "word_count"]].describe().T)

Twitter US Airline Sentiment: shape=(14640, 15)
Столбцы: ['tweet_id', 'airline_sentiment', 'airline_sentiment_confidence', 'negativereason', 'negativereason_confidence', 'airline', 'airline_sentiment_gold', 'name', 'negativereason_gold', 'retweet_count', 'text', 'tweet_coord', 'tweet_created', 'tweet_location', 'user_timezone']
Типы данных:


,dtype
tweet_id,int64
airline_sentiment,str
airline_sentiment_confidence,float64
negativereason,str
negativereason_confidence,float64
airline,str
airline_sentiment_gold,str
name,str
negativereason_gold,str
retweet_count,int64


Пропуски:


,missing
tweet_id,0
airline_sentiment,0
airline_sentiment_confidence,0
negativereason,5462
negativereason_confidence,4118
airline,0
airline_sentiment_gold,14600
name,0
negativereason_gold,14608
retweet_count,0


Первые 3 строки:


,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,@VirginAmerica What @dhepburn said.,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica plus you've added commercials t...,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,@VirginAmerica I didn't today... Must mean I n...,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)


Уникальные авиакомпании: 6
Распределение тональности:


,count
airline_sentiment,
negative,9178
neutral,3099
positive,2363


Минимальный числовой профиль текста и метаданных:


,count,mean,std,min,25%,50%,75%,max
airline_sentiment_confidence,14640.0,0.900169,0.162830,0.335,0.6923,1.0000,1.0,1.0
negativereason_confidence,10522.0,0.638298,0.330440,0.000,0.3606,0.6706,1.0,1.0
retweet_count,14640.0,0.082650,0.745778,0.000,0.0000,0.0000,0.0,44.0
text_length,14640.0,103.822063,36.277339,12.000,77.0000,114.0000,136.0,186.0
word_count,14640.0,17.653415,6.882259,2.000,12.0000,19.0000,23.0,36.0


В третьем фрагменте `tweet_id` и `name` относятся к номинальной шкале ID, `text` и координаты являются контентными признаками, `airline`, `airline_sentiment` и `negativereason` относятся к категориальной шкале, показатели уверенности и количество репостов относятся к шкале отношений, а `tweet_created` является интервальным признаком. Длина сообщения и количество слов являются производными числовыми признаками, которые будут использоваться при векторизации текста.

**Фрагмент 4. Поведение посетителей интернет-магазина**

В качестве четвёртого фрагмента выбран набор `Online Shoppers Purchasing Intention Dataset` из UCI Machine Learning Repository. Он содержит реальные сессии посетителей интернет-магазина, поведенческие характеристики, признаки источника трафика и бинарный признак завершения сессии покупкой. Источник получен из официального репозитория UCI программно. Его пригодность определяется непосредственной связью с анализом аудитории и конверсии: данные позволяют изучать, какие типы посетителей, источники трафика и поведенческие характеристики связаны с покупкой. При этом набор не содержит рекламных расходов и денежного дохода, поэтому не используется для расчёта ROAS и дополняет, а не заменяет основной финансовый источник.

In [5]:
openalex_url = (
    "https://api.openalex.org/works?search=social%20media%20marketing"
    "&per-page=50"
)
openalex_path = download_url(openalex_url, "openalex_social_media_marketing.json")
openalex_data = json.loads(openalex_path.read_text(encoding="utf-8"))
print("OpenAlex: тип корневого объекта:", type(openalex_data).__name__)
print("Ключи верхнего уровня:", list(openalex_data.keys()))
print("Количество результатов API:", len(openalex_data["results"]))

openalex_rows = []
for item in openalex_data["results"]:
    authorships = item.get("authorships") or []
    concepts = item.get("concepts") or []
    location = item.get("primary_location") or {}
    source = location.get("source") or {}
    openalex_rows.append({
        "work_id": item.get("id"),
        "title": item.get("title"),
        "publication_year": item.get("publication_year"),
        "type": item.get("type"),
        "is_oa": (item.get("open_access") or {}).get("is_oa"),
        "cited_by_count": item.get("cited_by_count"),
        "referenced_works_count": len(item.get("referenced_works") or []),
        "author_count": len(authorships),
        "concept_count": len(concepts),
        "source_type": source.get("type"),
        "source_name": source.get("display_name"),
        "title_length": len(item.get("title") or ""),
    })
openalex = pd.DataFrame(openalex_rows)
show_table_profile(openalex, "OpenAlex JSON API")
display(openalex[["publication_year", "cited_by_count", "referenced_works_count", "author_count", "concept_count", "title_length"]].describe().T)

OpenAlex: тип корневого объекта: dict
Ключи верхнего уровня: ['meta', 'results', 'group_by']
Количество результатов API: 50
OpenAlex JSON API: shape=(50, 12)
Столбцы: ['work_id', 'title', 'publication_year', 'type', 'is_oa', 'cited_by_count', 'referenced_works_count', 'author_count', 'concept_count', 'source_type', 'source_name', 'title_length']
Типы данных:


,dtype
work_id,str
title,str
publication_year,int64
type,str
is_oa,bool
cited_by_count,int64
referenced_works_count,int64
author_count,int64
concept_count,int64
source_type,str


Пропуски:


,missing
work_id,0
title,0
publication_year,0
type,0
is_oa,0
cited_by_count,0
referenced_works_count,0
author_count,0
concept_count,0
source_type,9


Первые 3 строки:


,work_id,title,publication_year,type,is_oa,cited_by_count,referenced_works_count,author_count,concept_count,source_type,source_name,title_length
0,https://openalex.org/W176499285,The Social Media Marketing Book,2009,article,False,533,0,1,24,NaN,NaN,31
1,https://openalex.org/W2269721265,Social media marketing and advertising,2015,article,True,402,61,3,9,journal,The Marketing Review,38
2,https://openalex.org/W2145416839,Foundations of Social Media Marketing,2014,conference-paper,True,318,89,1,8,journal,Procedia - Social and Behavioral Sciences,37


,count,mean,std,min,25%,50%,75%,max
publication_year,50.0,2014.60,3.591373,2008.0,2012.25,2014.0,2017.00,2024.0
cited_by_count,50.0,247.66,391.605154,17.0,52.50,111.5,369.00,2622.0
referenced_works_count,50.0,33.38,54.703613,0.0,0.00,15.0,50.00,332.0
author_count,50.0,2.32,2.486207,0.0,1.00,2.0,3.00,18.0
concept_count,50.0,14.50,5.296321,7.0,9.00,14.5,20.00,24.0
title_length,50.0,48.88,30.661842,22.0,22.00,40.0,58.25,137.0


In [6]:
uci_metadata_url = "https://archive.ics.uci.edu/api/dataset?id=468"
uci_data_url = "https://archive.ics.uci.edu/static/public/468/data.csv"
uci_metadata_path = download_url(uci_metadata_url, "uci_online_shoppers_metadata.json")
uci_data_path = download_url(uci_data_url, "uci_online_shoppers.csv")
uci_metadata = json.loads(uci_metadata_path.read_text(encoding="utf-8"))
shoppers = pd.read_csv(uci_data_path)
print("UCI metadata: корневые ключи:", list(uci_metadata.keys()))
print("UCI dataset:", uci_metadata.get("data", {}).get("name"))
print("UCI source-reported instances:", uci_metadata.get("data", {}).get("num_instances"))
show_table_profile(shoppers, "UCI Online Shoppers Purchasing Intention")
display(shoppers[["Administrative_Duration", "ProductRelated_Duration", "BounceRates", "ExitRates", "PageValues"]].describe().T)


UCI metadata: корневые ключи: ['status', 'statusText', 'data']
UCI dataset: Online Shoppers Purchasing Intention Dataset
UCI source-reported instances: 12330
UCI Online Shoppers Purchasing Intention: shape=(12330, 18)
Столбцы: ['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType', 'Weekend', 'Revenue']
Типы данных:


,dtype
Administrative,int64
Administrative_Duration,float64
Informational,int64
Informational_Duration,float64
ProductRelated,int64
ProductRelated_Duration,float64
BounceRates,float64
ExitRates,float64
PageValues,float64
SpecialDay,float64


Пропуски:


,missing
Administrative,0
Administrative_Duration,0
Informational,0
Informational_Duration,0
ProductRelated,0
ProductRelated_Duration,0
BounceRates,0
ExitRates,0
PageValues,0
SpecialDay,0


Первые 3 строки:


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.0,0.0,0.1,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False


,count,mean,std,min,25%,50%,75%,max
Administrative_Duration,12330.0,80.818611,176.779107,0.0,0.000000,7.500000,93.256250,3398.750000
ProductRelated_Duration,12330.0,1194.746220,1913.669288,0.0,184.137500,598.936905,1464.157214,63973.522230
BounceRates,12330.0,0.022191,0.048488,0.0,0.000000,0.003112,0.016813,0.200000
ExitRates,12330.0,0.043073,0.048597,0.0,0.014286,0.025156,0.050000,0.200000
PageValues,12330.0,5.889258,18.568437,0.0,0.000000,0.000000,0.000000,361.763742


В четвёртом фрагменте URL и домен относятся к номинальной шкале ID и контентной характеристике веб-ресурса, `first_path_segment`, суффикс URL и признак наличия параметров относятся к категориальным или дихотомическим признакам, а `lastmod` относится к интервальной шкале. Длина URL, глубина пути и число сегментов являются числовыми производными признаками абсолютной шкалы. XML используется как иерархическая структура веб-каталога.

**Сводная оценка качества и пригодности источников**

Проверка источников выполняется по фактическим файлам и ответам API. Для табличных данных выводятся форма, поля, типы, пропуски и примеры строк. Для JSON и XML дополнительно фиксируются тип корневого объекта, ключи или иерархические элементы и количество обработанных объектов. Полученные фрагменты не объединяются искусственно: общий анализ проводится на уровне отдельных аналитических таблиц, а слияние разрешается только для YouTube CSV и справочника категорий по `category_id`.

In [7]:
def dependency_check(df, determinant, dependent):
    grouped = df.groupby(determinant, dropna=False)[dependent].nunique(dropna=False)
    return {
        "determinant": ", ".join(determinant),
        "dependent": dependent,
        "groups": int(grouped.size),
        "violating_groups": int((grouped > 1).sum()),
        "holds_in_data": bool((grouped <= 1).all()),
    }

checks = [
    dependency_check(campaign, ["Ad Group", "Month"], "Cost"),
    dependency_check(campaign, ["Ad Group", "Month"], "Revenue"),
    dependency_check(youtube, ["video_id"], "title"),
    dependency_check(youtube, ["video_id"], "category_id"),
    dependency_check(airline, ["tweet_id"], "text"),
    dependency_check(airline, ["tweet_id"], "airline_sentiment"),
    dependency_check(openalex, ["work_id"], "title"),
    dependency_check(shoppers, ["Revenue"], "VisitorType"),
]
print("Проверка возможных функциональных зависимостей:")
display(pd.DataFrame(checks))

print("Дубликаты строк:")
for name, df in [("campaign", campaign), ("youtube", youtube), ("airline", airline), ("openalex", openalex), ("shoppers", shoppers)]:
    print(name, int(df.duplicated().sum()))

print("Средняя доля пропусков по источникам:")
for name, df in [("campaign", campaign), ("youtube", youtube), ("airline", airline), ("openalex", openalex), ("shoppers", shoppers)]:
    print(name, round(float(df.isna().mean().mean()), 4))

Проверка возможных функциональных зависимостей:


,determinant,dependent,groups,violating_groups,holds_in_data
0,"Ad Group, Month",Cost,190,0,True
1,"Ad Group, Month",Revenue,190,0,True
2,video_id,title,6351,109,False
3,video_id,category_id,6351,17,False
4,tweet_id,text,14485,0,True
5,tweet_id,airline_sentiment,14485,18,False
6,work_id,title,50,0,True
7,Revenue,VisitorType,2,2,False


Дубликаты строк:
campaign 0
youtube 48
airline 36
openalex 0
shoppers 125
Средняя доля пропусков по источникам:
campaign 0.0
youtube 0.0009
airline 0.2822
openalex 0.03


shoppers 0.0


Выбранные источники пригодны для дальнейшего исследования, поскольку покрывают финансовый, поведенческий, текстовый и тематический аспекты онлайн-маркетинга. Основной финансовый вывод будет строиться по данным рекламных кампаний. YouTube и Twitter используются для анализа закономерностей вовлечённости и пользовательской реакции. OpenAlex демонстрирует обработку API JSON, а UCI Online Shoppers характеризует поведение посетителей и факт покупки. Эти данные используются как дополнительные контекстные фрагменты, а не как источник ROAS.